Perfect 🚀 You want to design the **same agent framework** but this time **integrated with LangGraph** so that you can orchestrate the flow between LLM, MCP servers, and agents.

Here’s how we’ll structure it:

---

# 📂 Project Structure

```
order_mgmt_framework/
│── main.py
│
├── config/
│   ├── __init__.py
│   ├── settings.py
│
├── mcp_servers/
│   ├── __init__.py
│   ├── weather_server.py
│   ├── pollution_server.py
│
├── mcp_clients/
│   ├── __init__.py
│   ├── weather_client.py
│   ├── pollution_client.py
│
├── tools/
│   ├── __init__.py
│   ├── weather_tools.py
│   ├── pollution_tools.py
│
├── agents/
│   ├── __init__.py
│   ├── agent_factory.py
│   ├── parent_agent.py
```

---

# 🔹 `config/settings.py`

```python
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
DEFAULT_MODEL = "gpt-4o-mini"

AGENT_CONFIG = {
    "weather": {
        "tools": ["get_city_weather"],
        "mcp_servers": ["weather-mcp-1"]
    },
    "pollution": {
        "tools": ["get_city_pollution"],
        "mcp_servers": ["pollution-mcp-1"]
    },
    "parent": {
        "agents": ["weather", "pollution"],
        "llm_model": "gpt-4o-mini"
    }
}
```

---

# 🔹 `mcp_servers/weather_server.py`

```python
from fastmcp.server import FastMCP

WEATHER_DATA = {
    "Paris": "☁️ Cloudy, 22°C",
    "London": "🌧️ Rainy, 18°C",
    "Delhi": "☀️ Hot, 35°C"
}

mcp = FastMCP("weather-mcp-1")

@mcp.tool()
def get_city_weather(city: str):
    """Return weather info for a given city"""
    return {"content": WEATHER_DATA.get(city, "No weather data available")}

if __name__ == "__main__":
    print("Starting weather-mcp-1")
    mcp.run()
```

---

# 🔹 `mcp_servers/pollution_server.py`

```python
from fastmcp.server import FastMCP

POLLUTION_DATA = {
    "Delhi": "AQI 320 (Very Poor)",
    "Mumbai": "AQI 160 (Moderate)",
    "Paris": "AQI 70 (Good)"
}

mcp = FastMCP("pollution-mcp-1")

@mcp.tool()
def get_city_pollution(city: str):
    """Return pollution info for a given city"""
    return {"content": POLLUTION_DATA.get(city, "No pollution data available")}

if __name__ == "__main__":
    print("Starting pollution-mcp-1")
    mcp.run()
```

---

# 🔹 `mcp_clients/weather_client.py`

```python
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
import os

async def get_city_weather(city: str) -> str:
    """Client to call Weather MCP server"""
    current_dir = os.path.dirname(os.path.abspath(__file__))
    server_path = os.path.join(current_dir, "..", "mcp_servers", "weather_server.py")

    server_params = StdioServerParameters(
        command="python",
        args=[server_path]
    )

    async with stdio_client(server_params) as streams:
        async with ClientSession(*streams) as session:
            await session.initialize()
            result = await session.call_tool("get_city_weather", {"city": city})
            return result.content[0].text if result.content else None
```

---

# 🔹 `mcp_clients/pollution_client.py`

```python
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
import os

async def get_city_pollution(city: str) -> str:
    """Client to call Pollution MCP server"""
    current_dir = os.path.dirname(os.path.abspath(__file__))
    server_path = os.path.join(current_dir, "..", "mcp_servers", "pollution_server.py")

    server_params = StdioServerParameters(
        command="python",
        args=[server_path]
    )

    async with stdio_client(server_params) as streams:
        async with ClientSession(*streams) as session:
            await session.initialize()
            result = await session.call_tool("get_city_pollution", {"city": city})
            return result.content[0].text if result.content else None
```

---

# 🔹 `tools/weather_tools.py`

```python
import asyncio
from mcp_clients.weather_client import get_city_weather

class WeatherTools:
    @staticmethod
    def get_city_weather(city: str):
        return asyncio.run(get_city_weather(city))
```

---

# 🔹 `tools/pollution_tools.py`

```python
import asyncio
from mcp_clients.pollution_client import get_city_pollution

class PollutionTools:
    @staticmethod
    def get_city_pollution(city: str):
        return asyncio.run(get_city_pollution(city))
```

---

# 🔹 `agents/agent_factory.py`

```python
from tools.weather_tools import WeatherTools
from tools.pollution_tools import PollutionTools

class WeatherAgent:
    def run(self, city: str):
        return WeatherTools.get_city_weather(city)

class PollutionAgent:
    def run(self, city: str):
        return PollutionTools.get_city_pollution(city)

class AgentFactory:
    @staticmethod
    def create_agent(name: str):
        if name == "weather":
            return WeatherAgent()
        elif name == "pollution":
            return PollutionAgent()
        else:
            raise ValueError(f"Unknown agent: {name}")
```

---

# 🔹 `agents/parent_agent.py` (LangGraph)

```python
from langgraph.graph import StateGraph, END
from config.settings import AGENT_CONFIG
from agents.agent_factory import AgentFactory
from openai import OpenAI
import os

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# State Schema
class AgentState(dict):
    query: str
    intent: str
    city: str
    result: str

# --- Nodes ---
def classify_intent(state: AgentState) -> AgentState:
    """LLM decides weather or pollution + extract city"""
    prompt = f"""
    You are a classifier. Extract intent (weather/pollution) and city from query.
    User query: {state['query']}
    Respond in JSON: {{"intent": "...", "city": "..."}}.
    """
    resp = client.chat.completions.create(
        model=AGENT_CONFIG["parent"]["llm_model"],
        messages=[{"role": "user", "content": prompt}]
    )
    import json
    try:
        parsed = json.loads(resp.choices[0].message.content)
    except Exception:
        parsed = {"intent": "unknown", "city": ""}
    state["intent"] = parsed.get("intent")
    state["city"] = parsed.get("city")
    return state

def call_agent(state: AgentState) -> AgentState:
    """Route to the right child agent"""
    intent = state["intent"]
    city = state["city"]
    if intent in AGENT_CONFIG["parent"]["agents"]:
        agent = AgentFactory.create_agent(intent)
        state["result"] = agent.run(city)
    else:
        state["result"] = f"❌ Unknown intent: {intent}"
    return state

# --- Build Graph ---
workflow = StateGraph(AgentState)
workflow.add_node("classify", classify_intent)
workflow.add_node("call_agent", call_agent)

workflow.set_entry_point("classify")
workflow.add_edge("classify", "call_agent")
workflow.add_edge("call_agent", END)

app = workflow.compile()
```

---

# 🔹 `main.py`

```python
from agents.parent_agent import app

if __name__ == "__main__":
    print("🤖 Order Management Framework with LangGraph started!")
    while True:
        query = input("\nYou: ")
        if query.lower() in ["exit", "quit"]:
            break
        result = app.invoke({"query": query})
        print("Bot:", result["result"])
```

---

✅ Now the framework is **LangGraph-powered**:

* `classify_intent` node → LLM decides weather/pollution and extracts city.
* `call_agent` node → Routes to the correct MCP agent.
* Runs as a workflow.

---

Do you want me to also add **parallel execution** (so weather + pollution can be called together if user asks “Weather and pollution in Delhi”)?
